# STRATA Cross-Model Benchmark on Google Colab (T4 GPU)

Evaluates:
- **Qwen3.8-27B** (`unsloth/Qwen3.8-27B-GGUF`, UD-IQ3_S, 12.0 GB)
- **GPT-OSS 20B** (`unsloth/gpt-oss-20b-GGUF`, Q4_K_M, 11.6 GB)
- **Qwen3-Coder-30B-A3B** (`unsloth/Qwen3-Coder-30B-A3B-Instruct-GGUF`, Q3_K_S, 13.3 GB)
- **Qwen3.5 Highest T4 Fit**: `qwen3.5-35b-a3b` (35B MoE UD-IQ3_XXS, 13.1 GB) and `qwen3.5-27b` (27B Dense Q3_K_S, 12.3 GB)

Uses latest prebuilt `llama-server` (CUDA 12.8) from `ggml-org/llama.cpp` with automatic VRAM fallback and live streaming logs.

In [ ]:
# Cell 1: Verify Colab T4 GPU & cleanly clone STRATA (fix/strata-round2 until PR merged to dev)
!nvidia-smi
%cd /content
!rm -rf /content/STRATA
!git clone -b fix/strata-round2 https://github.com/MatthewNader2/STRATA.git /content/STRATA
%cd /content/STRATA
!pip -q install -r benchmarks/requirements-colab.txt

In [ ]:
# Cell 2: Download prebuilt CUDA llama-server + cudart cleanly with curl + integrity check
import json, os, pathlib, subprocess, urllib.request

req = urllib.request.Request(
    'https://api.github.com/repos/ggml-org/llama.cpp/releases?per_page=8',
    headers={'User-Agent': 'Mozilla/5.0'},
)
releases = json.loads(urllib.request.urlopen(req, timeout=20).read())
llama_dir = pathlib.Path('/content/llama_bin')
subprocess.run(['rm', '-rf', str(llama_dir)], check=True)
llama_dir.mkdir(parents=True, exist_ok=True)

chosen_assets = []
for rel in releases:
    assets = [a for a in rel.get('assets', []) if 'bin-ubuntu-cuda-12.8-x64.tar.gz' in a['name']]
    if any(a['name'].startswith('llama-') for a in assets) and any(a['name'].startswith('cudart-') for a in assets):
        chosen_assets = assets
        print('Using llama.cpp release:', rel['tag_name'])
        break
assert chosen_assets, 'No CUDA 12.8 Ubuntu llama.cpp release assets found!'

for asset in chosen_assets:
    dst = pathlib.Path('/content') / asset['name']
    ok_tar = dst.exists() and subprocess.run(['tar', '-tzf', str(dst)], capture_output=True).returncode == 0
    if not ok_tar:
        print('Downloading', asset['name'], '...')
        subprocess.run(['curl', '-fL', '--retry', '3', asset['browser_download_url'], '-o', str(dst)], check=True)
    subprocess.run(['tar', '-xzf', str(dst), '-C', str(llama_dir)], check=True)

server_bin = str(next(llama_dir.rglob('llama-server')))
so_dirs = sorted({str(p.parent) for p in llama_dir.rglob('*.so*')})
os.environ['LLAMA_SERVER'] = server_bin
os.environ['LD_LIBRARY_PATH'] = ':'.join(so_dirs + ['/usr/local/cuda/lib64', os.environ.get('LD_LIBRARY_PATH', '')]).rstrip(':')
print('LLAMA_SERVER =', os.environ['LLAMA_SERVER'])
!"$LLAMA_SERVER" --version

In [ ]:
# Cell 3: Run Core 9 Suites (8 presets + 3 CryoPulse tasks) with live streaming output
!python -u benchmarks/colab_eval.py \
  --llama-server "$LLAMA_SERVER" \
  --suite core9 \
  --models qwen3.8-27b gpt-oss-20b qwen3-coder-30b-a3b qwen3.5-35b-a3b qwen3.5-27b \
  --out-dir test_logs/colab_bench \
  --cleanup-cache \
  --resume

In [ ]:
# Cell 4 (Optional): Run Tier-2 Stress Suite (Aetheris + Realistic User + Honest Boundary)
!python -u benchmarks/colab_eval.py \
  --llama-server "$LLAMA_SERVER" \
  --suite stress \
  --models qwen3.8-27b gpt-oss-20b qwen3-coder-30b-a3b qwen3.5-35b-a3b qwen3.5-27b \
  --out-dir test_logs/colab_bench_stress \
  --cleanup-cache \
  --resume

In [ ]:
# Cell 5: Ingest Gemma 4 responses (if uploaded to /content/STRATA/gemma4_responses.txt), plot & zip results
import os
if os.path.exists('gemma4_responses.txt'):
    !python -u benchmarks/ingest_aistudio.py --input gemma4_responses.txt --suite all --out test_logs/colab_bench/results_gemma4_31b.json

!python -u benchmarks/visualize_results.py test_logs/colab_bench test_logs/colab_bench_stress --out-dir test_logs/visualizations
!zip -r /content/strata_colab_results.zip test_logs/colab_bench test_logs/colab_bench_stress test_logs/visualizations
try:
    from google.colab import files
    files.download('/content/strata_colab_results.zip')
except Exception:
    print('Download /content/strata_colab_results.zip manually from Colab file pane.')